# Module 5 Songs Data Cleaning and Analysis

Part A demonstrates cleaning on twelve fictional classroom songs. The cleaning
code is provided: run each small block and inspect what changes. You write and
test the `count_words` function. No loop is required.

Part B uses **nine real songs: three each by Taylor Swift, Olivia Dean, and BTS**.
Run its provided loading and cleaning cells, reuse your function, then write the
groupby summary, two plots, and explanations. Do not mix the two datasets.
Submit this notebook, `module05_songs.ipynb`, on Gradescope.

### Open it in VS Code

Keep this notebook beside `lyrics_api.py` and `data`. In the `05-wrangle` folder,
run `uv sync`, then select this repo's `.venv` Python as the notebook kernel.
The supplied API helper is not exam material. Use Codex for hints or feedback,
not completed functions or analysis.

### Internet and lyrics

Part A is offline. Part B needs internet to retrieve the same pinned LRCLIB
records for everyone. The supplied helper checks song, artist, album, and
duration; it stops with an explanation if a request fails or metadata changes.
Do not replace failed downloads with fictional data or different songs.
Lyrics stay in memory in a `lyrics` column. Do not display full lyrics in your
submitted notebook, commit them to GitHub, or paste them into your explanation.

## Part A Provided cleaning and your function


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from lyrics_api import load_demo_songs, load_homework_songs, load_homework_playlist

raw = load_demo_songs()
songs = raw.copy(deep=True)
display(songs[["song_id", "song", "artist", "duration_seconds", "status"]])


### Inspect the class data

One row is one playlist entry. These fictional songs contain original classroom
verses and invented durations; they are not measurements of real artists.
Inspect the types and preserve `raw` unchanged.

In [ ]:
display(songs.dtypes)
print("Rows:", len(songs))
print("Unique song IDs:", songs["song_id"].is_unique)


### Clean the artist names

The provided code creates a separate grouping column. It does not change the
original spelling or remove punctuation from names.

In [ ]:
songs["artist_clean"] = songs["artist"].str.strip().str.lower()
display(songs[["artist", "artist_clean"]])


### Convert playlist dates

This is instructor-created playlist information, not release dates from the
API. Different formats can represent the same date. We will merge by `song_id`.

In [ ]:
playlist_info = pd.DataFrame({
    "song_id": songs["song_id"],
    "playlist_group": ["commute", "focus", "commute", "focus", "focus", "commute", "focus", "commute", "commute", "focus", "focus", "commute"],
    "added_on": ["2026-09-01", "Sep 1, 2026", "2026-09-02", "September 2, 2026", "Sep 3, 2026", "2026-09-03", "2026-09-04", "Sep 4, 2026", "September 5, 2026", "2026-09-05", "Sep 6, 2026", "2026-09-06"],
})
display(playlist_info.head())


In [ ]:
print("Before:", playlist_info["added_on"].dtype)
playlist_info["added_on"] = pd.to_datetime(playlist_info["added_on"], format="mixed")
print("After:", playlist_info["added_on"].dtype)
print("Missing dates:", playlist_info["added_on"].isna().sum())
display(playlist_info.head())


### Inspect unavailable lyrics

An unavailable text is not a zero-word song. Keep these rows and the status
explaining why. Do not fill missing word counts with zero.

In [ ]:
missing = songs["lyrics"].isna()
display(songs.loc[missing, ["song_id", "song", "status"]])
print("Available lyrics:", songs["lyrics"].notna().sum())


## Write and test your word count function

Write `count_words(lyrics)`. It receives one string or missing value,
not a DataFrame row. Return `None` for missing or whitespace-only text;
otherwise return the number of whitespace-separated tokens. Use `pd.isna`,
string `split`, and `len` as needed. Punctuation stays attached to its token.

Test your function on `"blue train home"`, `"  blue   train  "`, `""`, and
`None`. Use separate function calls, not a loop. Predict each result before
running it. The same function will be reused for the real songs in Part B.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

### Apply your function to the class songs

After defining and testing `count_words`, remove the leading `#` from the next
two lines and run the cell. `map` calls your function once per lyrics value.

In [ ]:
# songs["word_count"] = songs["lyrics"].map(count_words)
# display(songs[["song", "word_count", "status"]])


### Merge the class tables

The provided left merge keeps all songs, including those without lyrics.
`validate="one_to_one"` checks that both sides have unique song IDs.

In [ ]:
class_enriched = songs.merge(playlist_info, on="song_id", how="left", validate="one_to_one")
print("Before:", len(songs), "After:", len(class_enriched))
display(class_enriched[["song_id", "song", "artist_clean", "added_on"]])


## Part B Homework groupby and plots

Now switch to the real-artist playlist. **Use `enriched` from the Part B cells
below**, not `songs` or `class_enriched` from Part A. The loader retrieves nine
pinned records, three per artist. BTS selections are English-language songs;
this makes whitespace counts more comparable, but the playlist still does not
represent any artist's entire catalog or measure lyrical quality.

Loading and cleaning are provided. You reuse your own `count_words` function
and write the summary and plotting code in the five homework tasks below.
A dashboard, Streamlit, and a fitted statistical model are not required.

### Load the real songs

Run this with internet access; nine requests can take up to a few minutes.
If it fails, restart the kernel and retry. If it still fails, contact the
instructor with the error rather than changing the playlist. Only metadata is
displayed; full lyrics remain in memory.

In [ ]:
homework_raw = load_homework_songs()
homework_playlist = load_homework_playlist()
display(homework_raw[["song_id", "song", "artist", "album", "duration_seconds", "status"]])


### Provided cleaning

Run these cells as written. Artist names become consistent grouping keys,
durations become numeric, and playlist dates become datetime values. The
original lyrics stay unchanged so your function can handle blank or missing
values itself. Playlist dates and groups are instructor-created metadata.

In [ ]:
homework_songs = homework_raw.copy(deep=True)
homework_songs["artist_clean"] = homework_songs["artist"].str.strip().str.lower()
homework_songs["duration_seconds"] = pd.to_numeric(homework_songs["duration_seconds"], errors="coerce")
homework_info = homework_playlist[["song_id", "playlist_group", "added_on"]].copy()
homework_info["added_on"] = pd.to_datetime(homework_info["added_on"], format="mixed")
enriched = homework_songs.merge(homework_info, on="song_id", how="left", validate="one_to_one")
print("Source rows:", len(homework_raw), "Prepared rows:", len(enriched))
display(enriched[["song_id", "song", "artist_clean", "duration_seconds", "added_on"]])


### Reuse your function

Remove the leading `#` from both lines after completing `count_words` in Part A.
Run this cell before Homework 1. It adds `word_count` but keeps the original
`lyrics` column. Do not display the whole `enriched` table: select only the
non-lyrics columns requested by the tasks.

In [ ]:
# enriched["word_count"] = enriched["lyrics"].map(count_words)
# display(enriched[["song_id", "song", "artist_clean", "word_count", "status"]])


## 1 Check your starting table

Use `enriched` from Part B, prepared by the supplied cleaning cells for
Taylor Swift, Olivia Dean, and BTS. First run the provided `map` line with your
own `count_words` function from Part A. Display song ID,
song, `artist_clean`, `duration_seconds`, `word_count`, `status`, and
`playlist_group`. Confirm nine rows, one per song ID, and the same row count as
`homework_raw`. Do not use the fictional `class_enriched` table.
Report how many word counts are missing. Keep those songs in the starting table;
missing word counts are not zero.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 2 Summarize by artist

Use `groupby` and `.agg` to create `artist_summary`, with one row per
`artist_clean` and these four columns:

- `total_songs`: number of songs, including songs without lyrics.
- `songs_with_text`: number of non-missing `word_count` values.
- `mean_words`: mean of the available word counts.
- `mean_duration_seconds`: mean duration of all songs with known duration.

Display the summary sorted from highest to lowest `mean_words`. Explain why
`total_songs` and `songs_with_text` can differ and which count is the denominator
for `mean_words`. Do not fill missing word counts before computing the mean.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 3 Plot the artist comparison

Create a bar chart from `artist_summary` showing `mean_words` for each
artist, ordered from highest to lowest. Label the artist axis and the mean word
count axis, and give the chart an informative title. Show `songs_with_text`
beside the chart in a small displayed table or in the chart labels.

Below it, identify the artist with the largest mean in this dataset. Cite the
mean and the number of available texts supporting it. Describe the selected
songs, not the artist's entire catalog or lyrical quality.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 4 Plot individual songs

Create a scatter plot from the song-level `enriched` table, not the
artist summary. Put `duration_seconds` on the horizontal axis and `word_count`
on the vertical axis. Use only rows where both values are known; report the
number of included and excluded songs. Label both axes and title the plot.

Describe whether longer durations consistently go with larger word counts in
these records. Cite two songs that support or complicate your observation.
Do not claim a causal relationship. Durations and lyrics come from a
user-contributed source, and nine selected songs cannot establish a general
music trend. Repetition contributes to the word count.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## 5 Explain and check your findings

Choose one artist. Display its song-level word counts and independently
check its mean using only the available values. Show the numerator and
denominator, then compare with your summary.

Write three to five sentences answering: What did the two plots help you see?
How could missing lyrics affect the comparison? Why does this small playlist
not represent an artist's entire catalog?

End with an assistance note: whether you used Codex, one hint you received,
and how you checked your work. If you did not use it, say so.

Before coding: predict what your result will look like.

In [ ]:
# Your code here


**Your check / explanation:** Replace this sentence with your own observation.

## Finish and submit

Submit one file, `module05_songs.ipynb`, to the **Module 5 homework
assignment on Gradescope**. Include the supplied cleaning cells, your function, homework
code, displayed summary, both plots, explanations, and assistance note.
Restart the kernel, run all cells in order, and save with outputs visible.
Part B requires internet to retrieve the pinned real-song records. If loading
fails repeatedly, contact the instructor; do not substitute the class demo.
Remove any outputs containing full downloaded lyrics before submitting; keep
the summaries, plots, and non-lyrics tables visible. No separate CSV, image,
or PDF upload is required.
Use the posted course calendar for the deadline. Do not push student work to
the shared repository.